In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("..") / "data" / "raw"
PROCESSED = Path("..") / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

meta = pd.read_csv(RAW / "_metadata.csv")

raw = {}
for sym in meta["symbol"]:
    raw[sym] = pd.read_csv(RAW / f"{sym}.csv", parse_dates=["Date"])

print(len(raw), "files loaded")

29 files loaded


In [2]:
clean = {}
removed = {}

for sym, df in raw.items():
    df = df.sort_values("Date").drop_duplicates("Date")
    if sym != "NIFTY50":
        fake = (df.Volume == 0) & (df.Open == df.High) & (df.High == df.Low) & (df.Low == df.Close)
        removed[sym] = int(fake.sum())
        df = df[~fake]
    clean[sym] = df.reset_index(drop=True)

print("Fake rows removed:", sum(removed.values()))
pd.Series(removed).value_counts()

Fake rows removed: 57


2    25
3     2
1     1
Name: count, dtype: int64

In [3]:
common = set.intersection(*[set(df["Date"]) for df in clean.values()])
print("Common trading days:", len(common))

aligned = {}
for sym, df in clean.items():
    dropped = (~df["Date"].isin(common)).sum()
    aligned[sym] = df[df["Date"].isin(common)].reset_index(drop=True)
    if dropped:
        print(sym, "dropped", dropped, "misaligned rows")

print("\nRows per file:", {len(df) for df in aligned.values()})

Common trading days: 1233
NIFTY50 dropped 3 misaligned rows
TCS dropped 2 misaligned rows
INFY dropped 2 misaligned rows
WIPRO dropped 2 misaligned rows
HCLTECH dropped 2 misaligned rows
HDFCBANK dropped 2 misaligned rows
ICICIBANK dropped 2 misaligned rows
SBIN dropped 2 misaligned rows
AXISBANK dropped 2 misaligned rows
KOTAKBANK dropped 2 misaligned rows
HINDUNILVR dropped 2 misaligned rows
ITC dropped 3 misaligned rows
NESTLEIND dropped 2 misaligned rows
SUNPHARMA dropped 2 misaligned rows
DRREDDY dropped 2 misaligned rows
CIPLA dropped 2 misaligned rows
MARUTI dropped 2 misaligned rows
MandM dropped 1 misaligned rows
BAJAJ_AUTO dropped 1 misaligned rows
RELIANCE dropped 2 misaligned rows
ONGC dropped 2 misaligned rows
NTPC dropped 2 misaligned rows
POWERGRID dropped 2 misaligned rows
TATASTEEL dropped 2 misaligned rows
HINDALCO dropped 2 misaligned rows
LT dropped 2 misaligned rows
ASIANPAINT dropped 2 misaligned rows
ULTRACEMCO dropped 2 misaligned rows
BHARTIARTL dropped 2 misal

In [4]:
for sym, df in aligned.items():
    df.to_csv(PROCESSED / f"{sym}.csv", index=False)

prices = pd.DataFrame({s: df.set_index("Date")["Close"] for s, df in aligned.items()})
returns = prices.pct_change().dropna()

prices.to_csv(PROCESSED / "prices_close.csv")
returns.to_csv(PROCESSED / "returns_daily.csv")

print("prices:", prices.shape, "| missing:", int(prices.isna().sum().sum()))
print("returns:", returns.shape)
returns.head()

prices: (1233, 29) | missing: 0
returns: (1232, 29)


,NIFTY50,TCS,INFY,WIPRO,HCLTECH,HDFCBANK,ICICIBANK,SBIN,AXISBANK,KOTAKBANK,...,RELIANCE,ONGC,NTPC,POWERGRID,TATASTEEL,HINDALCO,LT,ASIANPAINT,ULTRACEMCO,BHARTIARTL
Date,,,,,,,,,,,,,,,,,,,,,
2021-10-19,-0.003155,-0.003565,0.016740,0.002536,0.008761,0.011016,0.002683,-0.019580,-0.014324,0.011806,...,0.008956,-0.021591,0.000669,-0.025270,-0.031289,-0.016673,0.032207,-0.017816,-0.030976,0.000735
2021-10-20,-0.008261,-0.006962,-0.010866,-0.013492,0.000649,-0.008794,-0.002141,0.023966,0.003929,-0.009777,...,-0.011512,-0.023330,-0.022720,-0.021646,-0.017265,-0.038970,-0.021316,-0.004444,-0.005648,0.040156
2021-10-21,-0.004845,-0.021156,-0.027020,-0.008049,-0.009976,0.001464,0.012938,0.006101,0.003728,0.063658,...,-0.028848,0.000646,0.011282,-0.001543,-0.021291,-0.037431,-0.000526,-0.052877,-0.000084,-0.017503
2021-10-22,-0.003477,-0.009526,-0.019303,-0.019963,-0.013794,0.002655,0.005030,0.000000,0.011018,0.012898,...,0.001868,0.013226,-0.020284,0.002061,-0.014034,-0.047190,-0.007476,-0.006529,0.002890,-0.002227
2021-10-25,0.000580,-0.001686,-0.007123,-0.021542,-0.023953,-0.014131,0.108521,0.007058,0.034774,-0.007783,...,-0.009743,0.027698,-0.006901,-0.009257,-0.002276,0.005952,-0.004519,-0.021577,0.000028,-0.004896
